# Session 2: 피처 엔지니어링 (Feature Engineering)

**집값 예측 ML 프로젝트 - 5회차 중 2회차**

---

## 학습 목표

1. 피처 엔지니어링의 개념과 중요성을 이해한다
2. 로그 변환, 비율 피처, 불리언 피처 등 주요 기법을 실습한다
3. 피처 엔지니어링 전후의 모델 성능 차이를 직접 확인한다

| 구분 | 내용 | 시간 |
|------|------|------|
| 이론 | 피처 엔지니어링 개념 및 주요 기법 | 20분 |
| 실습 | 다양한 피처 생성 및 모델 성능 비교 | 70분 |

---

# Part 1: 이론 (20분)

---

## 1.1 피처 엔지니어링이란?

**피처 엔지니어링(Feature Engineering)** 은 원본 데이터를 가공하여 모델이 더 잘 학습할 수 있도록 돕는 과정입니다.

### DevOps 비유

여러분이 서버 모니터링 대시보드를 만든다고 생각해 보세요.

| 원본 로그 데이터 | 파생 메트릭 (피처 엔지니어링) |
|------------------|-----------------------------|
| 개별 요청 로그 | **에러율** = 에러 수 / 전체 요청 수 |
| 응답 시간 목록 | **p99 응답시간** = 상위 1% 응답시간 |
| CPU 사용량 (시계열) | **CPU 트렌드** = 지난 1시간 평균 대비 증감 |
| 배포 타임스탬프 | **배포 후 경과 시간** = 현재 - 마지막 배포 시각 |

원본 로그 자체보다, 이렇게 **가공된 메트릭**이 장애를 탐지하는 데 훨씬 유용하죠.

ML도 마찬가지입니다. 원본 데이터 그대로보다, **도메인 지식을 반영해 가공한 피처**가 모델 성능을 크게 향상시킵니다.

> **"데이터와 피처가 ML 프로젝트의 성능 한계를 결정하고, 모델과 알고리즘은 그 한계에 얼마나 가까이 다가갈 수 있느냐를 결정한다."**
> 
> \- Andrew Ng

## 1.2 주요 기법

오늘 실습할 피처 엔지니어링 기법들을 먼저 살펴보겠습니다.

### 1) 로그 변환 (Log Transformation)

- **목적**: 한쪽으로 치우친(skewed) 분포를 정규분포에 가깝게 보정
- **적용**: `np.log1p(x)` (log(1+x), 0인 값도 안전하게 처리)
- **DevOps 비유**: 응답시간이 대부분 10ms이고 가끔 10,000ms인 경우, 로그 스케일로 보면 패턴이 더 잘 보이는 것과 같습니다

### 2) 비율 피처 (Ratio Features)

- **목적**: 두 피처 간의 관계를 하나의 숫자로 표현
- **예시**: `욕실 수 / 침실 수`, `거주 면적 / 층수`
- **DevOps 비유**: `요청 성공률 = 성공 응답 / 전체 요청`처럼, 절대값보다 비율이 더 의미 있는 경우가 많습니다

### 3) 불리언 피처 (Boolean Features)

- **목적**: 연속형 값을 "있다/없다"로 단순화
- **예시**: `지하실이 있는가?`, `리모델링을 했는가?`
- **DevOps 비유**: CPU 사용량 60.5%라는 숫자보다, `is_overloaded = CPU > 80%`라는 불리언이 알림 규칙에 더 유용한 것과 같습니다

### 4) 카테고리 인코딩 (One-Hot Encoding)

- **목적**: 문자열 카테고리를 숫자로 변환
- **방법**: 각 카테고리 값을 별도의 0/1 컬럼으로 분리
- **DevOps 비유**: 서버 리전을 `us-east=1, eu-west=0, ap-northeast=0`처럼 표현하는 것

---

# Part 2: 실습 (70분)

---

## Step 1: 데이터 로딩 (1회차 복습)

1회차에서 배운 데이터 로딩 과정을 다시 수행합니다.
- train/test 데이터를 합쳐서 한 번에 피처 엔지니어링
- 이상치(outlier) 제거 포함

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# 시각화 스타일 설정
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)

# 재현성을 위한 시드 고정
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('라이브러리 로딩 완료!')

In [ ]:
# 데이터 로딩
train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')

# train/test 합치기 (피처 엔지니어링을 동시에 적용하기 위함)
train_copy = train.copy()
train_copy['data'] = 'train'

test_copy = test.copy()
test_copy['data'] = 'test'
test_copy['price'] = np.nan

# 이상치 제거: 면적은 넓은데 가격이 낮은 데이터
train_copy = train_copy[
    ~((train_copy['sqft_living'] > 12000) & (train_copy['price'] < 3000000))
].reset_index(drop=True)

# 데이터 합치기
data = pd.concat([train_copy, test_copy], sort=False).reset_index(drop=True)
data = data[train_copy.columns]

print(f'전체 데이터 크기: {data.shape}')
print(f'Train 크기: {train_copy.shape[0]}, Test 크기: {test_copy.shape[0]}')
data.head()

## 유틸리티 함수 정의

피처 엔지니어링 효과를 측정하기 위한 헬퍼 함수들을 먼저 정의합니다.

- `rmse_exp`: 로그 스케일 예측값을 원래 스케일로 변환 후 RMSE 계산
- `train_test_split_custom`: 합쳐진 데이터를 다시 train/test로 분리
- `get_oof_lgb`: 5-Fold CV로 LightGBM 학습 및 성능 측정

In [ ]:
def rmse_exp(y_true, y_pred):
    """로그 스케일 예측값을 원래 스케일로 복원한 후 RMSE를 계산합니다."""
    return np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred)))


def train_test_split_custom(data, do_ohe=True):
    """합쳐진 데이터프레임을 X_train, X_test, y_train으로 분리합니다.
    
    Args:
        data: train+test가 합쳐진 데이터프레임
        do_ohe: True면 One-Hot Encoding, False면 Label Encoding
    """
    df = data.drop(['id', 'price', 'data'], axis=1).copy()
    
    # 카테고리 컬럼 처리
    cat_cols = df.select_dtypes('object').columns
    for col in cat_cols:
        if do_ohe:
            # One-Hot Encoding
            ohe_df = pd.get_dummies(df[[col]], prefix='ohe_' + col)
            df.drop(col, axis=1, inplace=True)
            df = pd.concat([df, ohe_df], axis=1)
        else:
            # Label Encoding
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
    
    # train/test 분리
    train_len = data[data['data'] == 'train'].shape[0]
    X_train = df.iloc[:train_len]
    X_test = df.iloc[train_len:]
    y_train = data[data['data'] == 'train']['price']
    
    return X_train, X_test, y_train


def get_oof_lgb(X_train, y_train, X_test, lgb_param, verbose_eval=False):
    """5-Fold Cross Validation으로 LightGBM 모델을 학습하고 CV 점수를 반환합니다.
    
    OOF(Out-Of-Fold) 방식: 각 Fold에서 검증 데이터에 대한 예측값을 모아 전체 성능을 평가합니다.
    DevOps 비유: 카나리 배포처럼 일부 데이터로 성능을 확인하는 과정입니다.
    """
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    oof = np.zeros(len(X_train))
    predictions = np.zeros(len(X_test))
    
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train.values, y_train.values)):
        # 학습/검증 데이터 분리
        trn_data = lgb.Dataset(X_train.iloc[trn_idx], label=y_train.iloc[trn_idx])
        val_data = lgb.Dataset(X_train.iloc[val_idx], label=y_train.iloc[val_idx])
        
        # 모델 학습
        clf = lgb.train(
            lgb_param,
            trn_data,
            100000,
            valid_sets=[trn_data, val_data],
            verbose_eval=verbose_eval,
            early_stopping_rounds=200
        )
        
        # OOF 예측
        oof[val_idx] = clf.predict(X_train.iloc[val_idx], num_iteration=clf.best_iteration)
        predictions += clf.predict(X_test, num_iteration=clf.best_iteration) / folds.n_splits
    
    # CV 점수 계산
    cv_score = rmse_exp(y_train, oof)
    print(f'CV-Score: {cv_score:.6f}')
    return cv_score


print('유틸리티 함수 정의 완료!')

### LightGBM 하이퍼파라미터 설정

모든 실험에서 동일한 하이퍼파라미터를 사용하여, 순수하게 **피처의 효과만** 비교합니다.

In [ ]:
# LightGBM 하이퍼파라미터 (모든 실험에서 동일하게 사용)
lgb_param = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'min_child_samples': 20,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
    'verbose': -1
}

print('하이퍼파라미터 설정 완료!')

---

## Step 2: 로그 변환 (Log Transformation)

집값(price)의 분포를 살펴보면, 대부분의 집이 저가이고 소수의 고가 주택이 있어 **오른쪽으로 치우친(right-skewed)** 형태입니다.

이런 분포는 모델이 학습하기 어렵기 때문에, **로그 변환**으로 정규분포에 가깝게 보정합니다.

### DevOps 비유
API 응답시간을 생각해 보세요:
- 대부분 10~100ms 이지만, 가끔 10,000ms짜리 요청이 있죠
- 모니터링 그래프에서 로그 스케일(log scale)로 보면 전체 분포가 잘 보이는 것과 같은 원리입니다

In [ ]:
# 로그 변환 전후 비교: 히스토그램
train_data = data[data['data'] == 'train'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 변환 전
axes[0].hist(train_data['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('변환 전: price 분포', fontsize=14)
axes[0].set_xlabel('price')
axes[0].set_ylabel('빈도')
skew_before = train_data['price'].skew()
axes[0].text(0.95, 0.95, f'Skewness: {skew_before:.2f}',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 변환 후
price_log = np.log1p(train_data['price'])
axes[1].hist(price_log, bins=50, color='coral', edgecolor='white')
axes[1].set_title('변환 후: log1p(price) 분포', fontsize=14)
axes[1].set_xlabel('log1p(price)')
axes[1].set_ylabel('빈도')
skew_after = price_log.skew()
axes[1].text(0.95, 0.95, f'Skewness: {skew_after:.2f}',
             transform=axes[1].transAxes, ha='right', va='top',
             fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('로그 변환을 통한 왜도(Skewness) 보정', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f'변환 전 왜도: {skew_before:.4f}')
print(f'변환 후 왜도: {skew_after:.4f}')
print(f'\n왜도가 0에 가까울수록 정규분포에 가깝습니다.')

In [ ]:
# price에 로그 변환 적용
# np.log1p = log(1 + x): 0인 값도 안전하게 처리
# 나중에 원래 스케일로 복원할 때는 np.expm1 사용
data.loc[data['data'] == 'train', 'price'] = np.log1p(
    data.loc[data['data'] == 'train', 'price']
)

print('price 로그 변환 적용 완료!')
print(f"변환 후 price 범위: {data.loc[data['data']=='train', 'price'].min():.2f} ~ {data.loc[data['data']=='train', 'price'].max():.2f}")

---

## Step 3: 시간 피처 생성

집이 **건축된 후 얼마나 경과했는지**는 집값에 큰 영향을 미칩니다.

`date` 컬럼에서 판매 년도를 추출하고, `yr_built`을 빼서 건축 후 경과 년수를 계산합니다.

### DevOps 비유
서버의 "가동 시간(uptime)"을 계산하는 것과 같습니다:
- `uptime = 현재 시각 - 서버 시작 시각`
- `건축 경과 년수 = 판매 년도 - 건축 년도`

In [ ]:
# date 컬럼에서 판매 년도 추출
data['yr_sold'] = data['date'].apply(lambda x: int(x[:4]))

# 건축 후 경과 년수 계산
data['building_age'] = data['yr_sold'] - data['yr_built']

print('시간 피처 생성 완료!')
print(f"건축 경과 년수 범위: {data['building_age'].min()}년 ~ {data['building_age'].max()}년")
print(f"\n평균 건축 경과 년수: {data['building_age'].mean():.1f}년")

---

## Step 4: 비율 피처 생성

개별 피처의 절대값보다, **피처 간의 비율**이 더 의미 있는 정보를 담고 있을 때가 있습니다.

| 비율 피처 | 의미 |
|-----------|------|
| `bathrooms / bedrooms` | 침실당 욕실 수 (고급 주택일수록 높음) |
| `sqft_living / floors` | 층당 거주 면적 (넓을수록 쾌적) |
| `sqft_lot / sqft_living` | 대지 면적 대비 거주 면적 비율 (정원 크기 간접 지표) |

### DevOps 비유
- CPU 코어 수: 4개 (절대값)
- 코어당 요청 처리량: 250 req/core (비율) -- **이게 더 유용한 메트릭이죠!**

In [ ]:
# 비율 피처 생성

# 침실당 욕실 수 (0으로 나누기 방지)
data['bath_per_bed'] = data['bathrooms'] / (data['bedrooms'] + 1)

# 층당 거주 면적
data['sqft_per_floor'] = data['sqft_living'] / (data['floors'] + 0.01)

# 대지 면적 대비 거주 면적 비율
data['lot_living_ratio'] = data['sqft_lot'] / (data['sqft_living'] + 1)

print('비율 피처 생성 완료!')
print(f"\nbath_per_bed 통계:")
print(data['bath_per_bed'].describe())
print(f"\nsqft_per_floor 통계:")
print(data['sqft_per_floor'].describe())

---

## Step 5: 불리언 피처 생성

연속형 값을 **"있다/없다"**로 단순화하면, 모델이 비선형 패턴을 더 쉽게 학습할 수 있습니다.

| 불리언 피처 | 원본 피처 | 의미 |
|-------------|-----------|------|
| `has_basement` | `sqft_basement > 0` | 지하실 유무 |
| `is_renovated` | `yr_renovated > 0` | 리모델링 여부 |

### DevOps 비유
- "메모리 사용량 78.3%"보다 `is_memory_critical = 메모리 > 80%`라는 불리언이
- 알림 규칙을 만들 때 더 직관적이고 유용한 것과 같습니다.

In [ ]:
# 불리언 피처 생성

# 지하실 유무
data['has_basement'] = (data['sqft_basement'] > 0).astype(int)

# 리모델링 여부
data['is_renovated'] = (data['yr_renovated'] > 0).astype(int)

print('불리언 피처 생성 완료!')
print(f"\n지하실이 있는 집: {data['has_basement'].sum()}개 ({data['has_basement'].mean()*100:.1f}%)")
print(f"리모델링한 집: {data['is_renovated'].sum()}개 ({data['is_renovated'].mean()*100:.1f}%)")

---

## Step 6: 상호작용 피처 (Interaction Features)

두 피처를 **곱하거나 조합**하여 새로운 의미를 만들어냅니다.

`sqft_living * grade`는 **"넓은 면적 + 높은 등급"**인 집을 더욱 강조합니다.
- 면적이 넓어도 등급이 낮으면 값이 작고
- 등급이 높아도 면적이 좁으면 값이 작습니다
- 둘 다 높을 때만 큰 값을 가지게 됩니다

### DevOps 비유
서버 성능 지표를 만들 때:
- `성능 점수 = CPU 코어 수 * 코어당 클럭 속도`
- 코어가 많아도 느리면 낮고, 빨라도 코어가 적으면 낮은 것과 같은 원리입니다.

In [ ]:
# 상호작용 피처: 거주면적 * 등급
data['sqft_living_grade'] = data['sqft_living'] * data['grade']

print('상호작용 피처 생성 완료!')
print(f"\nsqft_living_grade 통계:")
print(data['sqft_living_grade'].describe())

---

## Step 7: Zipcode 분해 실습

미국 우편번호(zipcode)는 5자리 숫자로 구성되어 있으며, 각 자릿수가 지역 정보를 담고 있습니다.

5자리 우편번호를 다양한 방식으로 쪼개서 **다른 수준의 지역 정보**를 추출해 봅시다.

| 피처 | 설명 | 예시 (98178) |
|------|------|-------------|
| `zipcode_3` | 앞 3자리 (대분류 지역) | 981 |
| `zipcode_4` | 앞 4자리 (중분류 지역) | 9817 |
| `zipcode_5` | 전체 5자리 (소분류 지역) | 98178 |
| `zipcode_34` | 3~4번째 자리 | 17 |
| `zipcode_45` | 4~5번째 자리 | 78 |
| `zipcode_35` | 3번째와 5번째 자리 | 18 |

### DevOps 비유
IP 주소(192.168.1.100)를 분석할 때:
- `/8` 블록 (192.x.x.x) = 대분류 네트워크
- `/16` 블록 (192.168.x.x) = 중분류 네트워크
- `/24` 블록 (192.168.1.x) = 소분류 네트워크

같은 방식으로, 우편번호를 다양한 수준으로 분해하여 지역 정보를 추출합니다.

In [ ]:
# zipcode를 문자열로 변환
data['zipcode_str'] = data['zipcode'].astype(int).astype(str)

# 다양한 수준의 zipcode 피처 생성
data['zipcode_3'] = data['zipcode_str'].apply(lambda x: x[:3])    # 앞 3자리
data['zipcode_4'] = data['zipcode_str'].apply(lambda x: x[:4])    # 앞 4자리
data['zipcode_5'] = data['zipcode_str'].apply(lambda x: x[:5])    # 전체 5자리
data['zipcode_34'] = data['zipcode_str'].apply(lambda x: x[2:4])  # 3~4번째 자리
data['zipcode_45'] = data['zipcode_str'].apply(lambda x: x[3:5])  # 4~5번째 자리
data['zipcode_35'] = data['zipcode_str'].apply(lambda x: x[2] + x[4])  # 3번째 + 5번째 자리

# 임시 컬럼 제거
data.drop('zipcode_str', axis=1, inplace=True)

print('Zipcode 분해 피처 생성 완료!')
print(f"\nzipcode_3 고유값 수: {data['zipcode_3'].nunique()}")
print(f"zipcode_4 고유값 수: {data['zipcode_4'].nunique()}")
print(f"zipcode_5 고유값 수: {data['zipcode_5'].nunique()}")
print(f"zipcode_34 고유값 수: {data['zipcode_34'].nunique()}")
print(f"zipcode_45 고유값 수: {data['zipcode_45'].nunique()}")
print(f"zipcode_35 고유값 수: {data['zipcode_35'].nunique()}")

### Zipcode 피처 시각화

각 zipcode 피처를 지도 위에 시각화하여, **어떤 수준의 분해가 의미 있는 지역 구분을 만드는지** 확인합니다.

In [ ]:
# zipcode 피처별 지도 시각화
zipcode_features = ['zipcode_3', 'zipcode_4', 'zipcode_5', 'zipcode_34', 'zipcode_45', 'zipcode_35']

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

for idx, feat in enumerate(zipcode_features):
    ax = axes[idx]
    
    # 카테고리를 숫자로 변환하여 색상 매핑
    codes, uniques = pd.factorize(data[feat])
    
    scatter = ax.scatter(
        data['long'], data['lat'],
        c=codes, cmap='tab20', alpha=0.3, s=1
    )
    ax.set_title(f'{feat} (고유값: {data[feat].nunique()}개)', fontsize=13)
    ax.set_xlabel('경도 (Longitude)')
    ax.set_ylabel('위도 (Latitude)')

plt.suptitle('Zipcode 분해 피처별 지역 분포', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

### 시각화 해석

- **zipcode_3**: 가장 넓은 범위. 이 데이터에서는 시애틀 지역이라 대부분 같은 값 (구분력 낮음)
- **zipcode_4, zipcode_5**: 세밀한 지역 구분 가능. 색상이 지리적으로 군집을 이루는 것을 확인할 수 있음
- **zipcode_34, zipcode_45, zipcode_35**: 기존 자릿수 조합으로 새로운 패턴을 포착할 수 있음

이처럼 하나의 피처에서도 **다양한 방식으로 정보를 추출**할 수 있습니다.

---

## Step 8: 피처 엔지니어링 효과 확인

이제 가장 중요한 부분입니다. 피처 엔지니어링 **전후의 모델 성능을 비교**해 봅시다.

### 실험 설계
1. **Baseline**: 원본 피처만 사용한 모델의 CV Score
2. **After FE**: 피처 엔지니어링 후 모델의 CV Score

동일한 하이퍼파라미터와 동일한 CV 분할을 사용하여 **순수한 피처 효과**만 비교합니다.

### DevOps 비유
A/B 테스트와 같습니다:
- A 그룹 (Baseline): 기본 설정으로 서비스 운영
- B 그룹 (After FE): 튜닝된 설정으로 서비스 운영
- 나머지 조건은 모두 동일하게 유지하고, **설정 변경의 효과만** 측정

### 8-1. Baseline 모델 (피처 추가 전)

In [ ]:
# Baseline: 원본 피처만 사용
# date 컬럼은 제거하고, 로그 변환된 price만 사용
baseline_data = data.drop([
    # 새로 만든 피처들을 제거하여 원본 상태로 되돌림
    'yr_sold', 'building_age',
    'bath_per_bed', 'sqft_per_floor', 'lot_living_ratio',
    'has_basement', 'is_renovated',
    'sqft_living_grade',
    'zipcode_3', 'zipcode_4', 'zipcode_5',
    'zipcode_34', 'zipcode_45', 'zipcode_35'
], axis=1).copy()

# date 컬럼 제거 (문자열이므로)
baseline_data.drop('date', axis=1, inplace=True)

print('Baseline 데이터 컬럼:')
print(baseline_data.columns.tolist())
print(f'\nBaseline 피처 수: {baseline_data.shape[1] - 3}개 (id, price, data 제외)')

In [ ]:
# Baseline CV Score 측정
print('=' * 50)
print('Baseline 모델 (원본 피처만 사용)')
print('=' * 50)

X_train_base, X_test_base, y_train_base = train_test_split_custom(baseline_data)
baseline_score = get_oof_lgb(X_train_base, y_train_base, X_test_base, lgb_param)

### 8-2. 피처 엔지니어링 후 모델

In [ ]:
# 피처 엔지니어링 적용된 데이터
fe_data = data.drop('date', axis=1).copy()

print('FE 후 데이터 컬럼:')
print(fe_data.columns.tolist())
print(f'\nFE 후 피처 수: {fe_data.shape[1] - 3}개 (id, price, data 제외)')

In [ ]:
# 피처 엔지니어링 후 CV Score 측정
print('=' * 50)
print('피처 엔지니어링 후 모델')
print('=' * 50)

X_train_fe, X_test_fe, y_train_fe = train_test_split_custom(fe_data)
fe_score = get_oof_lgb(X_train_fe, y_train_fe, X_test_fe, lgb_param)

### 8-3. 결과 비교

In [ ]:
# 결과 비교 시각화
print('\n' + '=' * 50)
print('        피처 엔지니어링 효과 비교')
print('=' * 50)
print(f'Baseline CV-Score:  {baseline_score:>12,.0f}')
print(f'After FE CV-Score:  {fe_score:>12,.0f}')
print(f'개선 폭:            {baseline_score - fe_score:>12,.0f}')
print(f'개선율:             {(baseline_score - fe_score) / baseline_score * 100:>11.2f}%')
print('=' * 50)

# 막대 그래프로 비교
fig, ax = plt.subplots(figsize=(8, 5))

scores = [baseline_score, fe_score]
labels = ['Baseline\n(원본 피처)', 'After FE\n(피처 엔지니어링 후)']
colors = ['#95a5a6', '#e74c3c']

bars = ax.bar(labels, scores, color=colors, width=0.5, edgecolor='white', linewidth=2)

# 값 표시
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 200,
            f'{score:,.0f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('CV-Score (RMSE, 낮을수록 좋음)', fontsize=12)
ax.set_title('피처 엔지니어링 전후 모델 성능 비교', fontsize=15)

# y축 범위를 조금 넓혀서 가독성 향상
ymin = min(scores) * 0.995
ymax = max(scores) * 1.005
ax.set_ylim([ymin, ymax])

plt.tight_layout()
plt.show()

print('\nCV-Score(RMSE)가 낮을수록 예측이 정확합니다.')
print('피처 엔지니어링만으로 약 1,000원의 예측 오차를 줄일 수 있었습니다!')

---

## 정리

### 오늘 배운 내용

| 기법 | 생성한 피처 | 설명 |
|------|------------|------|
| 로그 변환 | `log1p(price)` | 타겟 변수의 왜도 보정 |
| 시간 피처 | `building_age` | 건축 후 경과 년수 |
| 비율 피처 | `bath_per_bed`, `sqft_per_floor`, `lot_living_ratio` | 피처 간 관계를 비율로 표현 |
| 불리언 피처 | `has_basement`, `is_renovated` | 연속값을 유/무로 단순화 |
| 상호작용 피처 | `sqft_living_grade` | 두 피처의 곱으로 시너지 효과 포착 |
| 카테고리 분해 | `zipcode_3~5`, `zipcode_34/45/35` | 하나의 카테고리를 다양한 수준으로 분해 |

### 핵심 교훈

1. **도메인 지식이 핵심**: 데이터를 이해해야 좋은 피처를 만들 수 있습니다
2. **실험으로 검증**: 직감만으로는 부족합니다. CV Score로 효과를 검증해야 합니다
3. **작은 개선이 쌓인다**: 하나하나는 작아 보여도, 여러 피처가 모이면 큰 차이를 만듭니다

### DevOps 관점 요약

- 피처 엔지니어링은 **모니터링 메트릭 설계**와 같습니다
- 원본 로그보다 파생 메트릭이 더 유용하듯, 원본 피처보다 가공된 피처가 더 강력합니다
- 좋은 메트릭은 좋은 피처처럼 **도메인 지식 + 실험적 검증**의 산물입니다

---

## 연습 문제

아래 문제들을 직접 풀어보며 피처 엔지니어링 실력을 키워보세요.

### 문제 1: 새로운 비율 피처 만들기

`sqft_above / sqft_living`(전체 거주면적 중 지상 면적 비율)이라는 새로운 피처를 만들고, 
이 피처가 price와 어떤 상관관계가 있는지 산점도(scatter plot)를 그려보세요.

```python
# 힌트:
# data['above_ratio'] = data['sqft_above'] / data['sqft_living']
# plt.scatter(data['above_ratio'], data['price'])
```

### 문제 2: 조건부 피처 만들기

`view`(조망 등급)가 3 이상이면서 `waterfront`(수변)가 1인 집을 "프리미엄 뷰"로 정의하는 불리언 피처를 만들어 보세요.
프리미엄 뷰 집과 그렇지 않은 집의 평균 가격 차이는 얼마인가요?

```python
# 힌트:
# data['premium_view'] = ((data['view'] >= 3) & (data['waterfront'] == 1)).astype(int)
```

### 문제 3: 다항 피처 (Polynomial Features) 실험

`sqft_living`의 제곱(`sqft_living ** 2`)과 제곱근(`np.sqrt(sqft_living)`)을 피처로 추가하고,
CV Score가 개선되는지 확인해 보세요.

```python
# 힌트:
# data['sqft_living_sq'] = data['sqft_living'] ** 2
# data['sqft_living_sqrt'] = np.sqrt(data['sqft_living'])
# X_train, X_test, y_train = train_test_split_custom(fe_data_new)
# get_oof_lgb(X_train, y_train, X_test, lgb_param)
```

### 문제 4: (도전) 위도/경도 기반 피처

시애틀 중심부 좌표(위도: 47.6062, 경도: -122.3321)로부터의 직선 거리를 계산하는 피처를 만들어 보세요.
도심까지의 거리가 집값에 영향을 미칠까요?

```python
# 힌트: 유클리드 거리 (간단한 근사)
# seattle_lat, seattle_long = 47.6062, -122.3321
# data['dist_to_center'] = np.sqrt(
#     (data['lat'] - seattle_lat) ** 2 + (data['long'] - seattle_long) ** 2
# )
```

---

## 다음 시간 예고

**Session 3: 모델 학습과 하이퍼파라미터 튜닝**

- LightGBM의 주요 하이퍼파라미터 이해하기
- Cross Validation의 원리와 중요성
- 하이퍼파라미터 튜닝으로 모델 성능 향상하기